In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.io as pio

pio.renderers.default = "browser"
%matplotlib inline
pd.options.display.float_format = '{:,.2f}'.format

In [ ]:
df_data = pd.read_csv('nobel_prize_data.csv')

print(f"Dataset Shape: {df_data.shape}")
print(f"Any duplicates? {df_data.duplicated().values.any()}")
print(f"Any NaN values among the data? {df_data.isna().values.any()}\n")

print("Missing values per column:")
print(df_data.isna().sum())


In [ ]:
df_data.birth_date = pd.to_datetime(df_data.birth_date)

separated_values = df_data.prize_share.str.split('/', expand=True)
numerator = pd.to_numeric(separated_values[0])
denominator = pd.to_numeric(separated_values[1])
df_data['share_pct'] = numerator / denominator

birth_years = df_data.birth_date.dt.year
df_data['winning_age'] = df_data.year - birth_years

df_data.info()

In [ ]:
biology = df_data.sex.value_counts()
fig = px.pie(
    values=biology.values,
    names=biology.index,
    title="Percentage of Male vs. Female Winners",
    hole=0.4
)
fig.update_traces(textposition='inside', textfont_size=15, textinfo='percent')
fig.show()
print("Earliest Female Nobel Laureates:")
display(df_data[df_data.sex == 'Female'].sort_values('year', ascending=True)[:3])

multiple_winners = df_data.groupby(by='full_name').filter(lambda x: x['year'].count() >= 2)
print(f'\nThere are {multiple_winners.full_name.nunique()} winners who were awarded the prize more than once.')
multiple_winners[['year', 'category', 'laureate_type', 'full_name']]

In [ ]:
prizes_per_category = df_data.category.value_counts()
v_bar = px.bar(
    x=prizes_per_category.index,
    y=prizes_per_category.values,
    color=prizes_per_category.values,
    color_continuous_scale='Aggrnyl',
    title='Number of Prizes Awarded per Category'
)
v_bar.update_layout(xaxis_title='Nobel Prize Category', coloraxis_showscale=False, yaxis_title='Number of Prizes')
v_bar.show()

cat_men_women = df_data.groupby(['category', 'sex'], as_index=False).agg({'prize': pd.Series.count})
cat_men_women.sort_values('prize', ascending=False, inplace=True)

v_bar_split = px.bar(
    x=cat_men_women.category,
    y=cat_men_women.prize,
    color=cat_men_women.sex,
    title='Number of Prizes Awarded per Category split by Men and Women'
)
v_bar_split.update_layout(xaxis_title='Nobel Prize Category', yaxis_title='Number of Prizes')
v_bar_split.show()

In [ ]:
prize_per_year = df_data.groupby(by='year').count().prize
moving_average = prize_per_year.rolling(window=5).mean()

yearly_avg_share = df_data.groupby(by='year').agg({'share_pct': pd.Series.mean})
share_moving_average = yearly_avg_share.rolling(window=5).mean()

plt.figure(figsize=(16, 8), dpi=200)
plt.title('Number of Nobel Prizes Awarded per Year', fontsize=18)
plt.yticks(fontsize=14)
plt.xticks(ticks=np.arange(1900, 2021, step=5), fontsize=14, rotation=45)

ax1 = plt.gca()
ax2 = ax1.twinx()  
ax1.set_xlim(1900, 2020)
ax2.invert_yaxis()  

ax1.scatter(x=prize_per_year.index, y=prize_per_year.values, c='dodgerblue', alpha=0.7, s=100)
ax1.plot(prize_per_year.index, moving_average.values, c='crimson', linewidth=3)
ax2.plot(prize_per_year.index, share_moving_average.values, c='grey', linewidth=3)
plt.show()

In [ ]:
top_countries = df_data.groupby(['birth_country_current'], as_index=False).agg({'prize': pd.Series.count})
top_countries.sort_values(by='prize', inplace=True)
top20_countries = top_countries[-20:]

h_bar = px.bar(
    x=top20_countries.prize,
    y=top20_countries.birth_country_current,
    orientation='h',
    color=top20_countries.prize,
    color_continuous_scale='Viridis',
    title='Top 20 Countries by Number of Prizes'
)
h_bar.update_layout(xaxis_title='Number of Prizes', yaxis_title='Country', coloraxis_showscale=False)
h_bar.show()

df_countries = df_data.groupby(['birth_country_current', 'ISO'], as_index=False).agg({'prize': pd.Series.count})
world_map = px.choropleth(
    df_countries,
    locations='ISO',
    color='prize', 
    hover_name='birth_country_current', 
    color_continuous_scale=px.colors.sequential.matter,
    title='Global Map Distribution of Nobel Laureates'
)
world_map.show()

prize_by_year = df_data.groupby(by=['birth_country_current', 'year'], as_index=False).count()
prize_by_year = prize_by_year.sort_values('year')[['year', 'birth_country_current', 'prize']]
cumulative_prizes = prize_by_year.groupby(by=['birth_country_current', 'year']).sum().groupby(level=[0]).cumsum().reset_index()

l_chart = px.line(
    cumulative_prizes,
    x='year', 
    y='prize',
    color='birth_country_current',
    title='Cumulative Nobel Prizes Over Time by Country'
)
l_chart.update_layout(xaxis_title='Year', yaxis_title='Number of Prizes')
l_chart.show()

In [ ]:
country_city_org = df_data.groupby(
    by=['organization_country', 'organization_city', 'organization_name'], 
    as_index=False
).agg({'prize': pd.Series.count})
country_city_org = country_city_org.sort_values('prize', ascending=False)

burst = px.sunburst(
    country_city_org, 
    path=['organization_country', 'organization_city', 'organization_name'], 
    values='prize',
    title='Where do Discoveries Take Place?'
)
burst.show()

In [ ]:
print("Winning Age Descriptive Statistics:")
print(df_data.winning_age.describe())

box = px.box(
    df_data, 
    x='category', 
    y='winning_age',
    title='How old are the Winners?'
)
box.update_layout(xaxis_title='Category', yaxis_title='Age at time of Award', xaxis={'categoryorder':'mean ascending'})
box.show()

sns.set_theme(style='whitegrid')
sns.lmplot(
    data=df_data,
    x='year', 
    y='winning_age',
    row='category',
    lowess=True, 
    aspect=2,
    scatter_kws={'alpha': 0.6},
    line_kws={'color': 'black'}
)
plt.show()